# EECS 230 Deep Learning - Assignment 2
Due at **11:59PM March 1st, 2024**

Put your answers in solution sections, add missing code, run all code blocks.

Make sure you choose a runtime type with a GPU, e.g., T4 GPU. You don't need to subscrible to Colab Pro for such runtime type.

What to submit: Print the notebook with output, save as a PDF, and submit the PDF to CatCourse.

It is very important you have output shown in the PDF submitted.

Save a copy of this notebook in drive so you don't lose your work.

## Part I: Convolutional Neural Network (3 points)
1. [2 points] Consider the convolutional neural network defined by the layers in the left column below. Fill in the shape of the output volume and the number of parameters at each layer. You can write the activation shapes in the format (H, W, C) for 3D volume, where H, W, C are the height, width and channel dimensions, respectively. Unless specified, assume padding 1, stride 1 where appropriate.

Notation:
<ul>
<li>CONVx-N denotes a convolutional layer with N filters with kernel height and width equal to x</li>
<li>POOL-N denotes a N×N max-pooling layer with stride of N and 0 padding.</li>
<li>FLATTEN flattens its inputs, identical to torch.nn.flatten / tf.layers.flatten </li>
<li>FC-N denotes a fully-connected layer with N neurons</li>
</ul>

Fill out the table bellow:

<font color="red">Soluton: </font>

| Layer      | Activation volume dimensions | Number of parameters (weights and biases)     |
| :---        |    :----:   |          ---: |
| Input      | (32, 32, 3)       | 0   |
| Conv3-8   |         |       |
| ReLU   |         |       |
| POOL-2   |         |       |
| BATCHNORM   |         |       |
| Conv3-16   |         |       |
| ReLU   |         |       |
| POOL-2   |         |       |
| FLATTEN   |         |       |
| FC-10  |         |       |

2. [1 point] What types of losses are used to train an object detection network? Notation is not needed here. Just describe the losses in two to four sentences.

<font color="red">Soluton: </font>

3. [1 point] What is the goal of dilated convolution (a.k.a. atrous convolution)? Notation is not needed here. Answer in two to four sentences.

<font color="red">Soluton: </font>

## Part II: Open-vocabulary Image Classification (3 points)

4. In this part, you are supposed to use CLIP model to build an open-vocabulary image classification.

Related lecture: https://ucmercedeecs230.github.io/files/lec08_2.pdf

### Preparation for Colab

Make sure you're running a GPU runtime; if not, select "GPU" as the hardware accelerator in Runtime > Change Runtime Type in the menu. The next cells will install the `clip` package and its dependencies, and check if PyTorch 1.7.1 or later is installed.

In [ ]:
! pip install ftfy regex tqdm
! pip install git+https://github.com/openai/CLIP.git

In [ ]:
import numpy as np
import torch
from pkg_resources import packaging

print("Torch version:", torch.__version__)

### Loading the model

`clip.available_models()` will list the names of available CLIP models.

In [ ]:
import clip

clip.available_models()

In [ ]:
# We will use CLIP with ViT-B/32 backbone in this assignment
model, preprocess = clip.load("ViT-B/32")
model.cuda().eval()
input_resolution = model.visual.input_resolution
context_length = model.context_length
vocab_size = model.vocab_size

print("Model parameters:", f"{np.sum([int(np.prod(p.shape)) for p in model.parameters()]):,}")
print("Input resolution:", input_resolution)
print("Context length:", context_length)
print("Vocab size:", vocab_size)

### Image Preprocessing

The second return value from `clip.load()` contains a torchvision `Transform` that performs this preprocessing.

In [ ]:
preprocess

<font color="red">4.1 [1 point] What are the image processing steps included in image preprocess above?</font>

<font color="red">Soluton: </font>

### Text Preprocessing

We use a case-insensitive tokenizer, which can be invoked using `clip.tokenize()`. By default, the outputs are padded to become 77 tokens long, which is what the CLIP models expects.

In [ ]:
clip.tokenize("Hello World!")

### Setting up input images and texts

We are going to feed 8 example images and their textual descriptions to the model, and compare the similarity between the corresponding features.

The tokenizer is case-insensitive, and we can freely give any suitable textual descriptions.

In [ ]:
import os
import skimage
import IPython.display
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np

from collections import OrderedDict
import torch

%matplotlib inline
%config InlineBackend.figure_format = 'retina'

# images in skimage to use and their textual descriptions
descriptions = {
    "page": "a page of text about segmentation",
    "chelsea": "a facial photo of a tabby cat",
    "astronaut": "a portrait of an astronaut with the American flag",
    "rocket": "a rocket standing on a launchpad",
    "motorcycle_right": "a red motorcycle standing in a garage",
    "camera": "a person looking at a camera on a tripod",
    "horse": "a black-and-white silhouette of a horse",
    "coffee": "a cup of coffee on a saucer"
}

original_images = []
images = []
texts = []
plt.figure(figsize=(16, 5))

for filename in [filename for filename in os.listdir(skimage.data_dir) if filename.endswith(".png") or filename.endswith(".jpg")]:
    name = os.path.splitext(filename)[0]
    if name not in descriptions:
        continue

    image = Image.open(os.path.join(skimage.data_dir, filename)).convert("RGB")

    plt.subplot(2, 4, len(images) + 1)
    plt.imshow(image)
    plt.title(f"{filename}\n{descriptions[name]}")
    plt.xticks([])
    plt.yticks([])

    original_images.append(image)
    images.append(preprocess(image))
    texts.append(descriptions[name])

plt.tight_layout()

### Building features

We normalize the images, tokenize each text input, and run the forward pass of the model to get the image and text features.

In [ ]:
image_input = torch.tensor(np.stack(images)).cuda()
text_tokens = clip.tokenize(["This is " + desc for desc in texts]).cuda()

In [ ]:
with torch.no_grad():
    image_features = model.encode_image(image_input).float()
    text_features = model.encode_text(text_tokens).float()

<font color="red">4.2 [1 point] Fill missing code bellow to compute similarity between image embedding and text embedding. First normalize image features and text features to be unit norm. Then take the dot product between text features and image faetures.</font>

### Calculating cosine similarity

We normalize the features and calculate the dot product of each pair.

In [ ]:
image_features /=  # complete code here
text_features /=  # complete code here
similarity =  # complete code here

count = len(descriptions)

plt.figure(figsize=(20, 14))
plt.imshow(similarity, vmin=0.1, vmax=0.3)
# plt.colorbar()
plt.yticks(range(count), texts, fontsize=18)
plt.xticks([])
for i, image in enumerate(original_images):
    plt.imshow(image, extent=(i - 0.5, i + 0.5, -1.6, -0.6), origin="lower")
for x in range(similarity.shape[1]):
    for y in range(similarity.shape[0]):
        plt.text(x, y, f"{similarity[y, x]:.2f}", ha="center", va="center", size=12)

for side in ["left", "top", "right", "bottom"]:
  plt.gca().spines[side].set_visible(False)

plt.xlim([-0.5, count - 0.5])
plt.ylim([count + 0.5, -2])

plt.title("Cosine similarity between text and image features", size=20)

### Zero-Shot Image Classification

You can classify images using the cosine similarity (times 100) as the logits to the softmax operation.

In [ ]:
from torchvision.datasets import CIFAR100

cifar100 = CIFAR100(os.path.expanduser("~/.cache"), transform=preprocess, download=True)

<font color="red">4.3 [1 point] Complete code bellow for zero-shot image classification</font>

In [ ]:
text_descriptions = [f"This is a photo of a {label}" for label in cifar100.classes]

# Tokenize text descriptions
text_tokens =  #complete here
with torch.no_grad():
    #encode text
    text_features = #complete here
    #normalized text features
    text_features /= #complete here
#compute dot product between pairs of image feature and text feature
similarity = #complete here

text_probs = (100.0 * similarity).softmax(dim=-1)
top_probs, top_labels = text_probs.cpu().topk(5, dim=-1)

In [ ]:
# visualize classification results
plt.figure(figsize=(16, 16))

for i, image in enumerate(original_images):
    plt.subplot(4, 4, 2 * i + 1)
    plt.imshow(image)
    plt.axis("off")

    plt.subplot(4, 4, 2 * i + 2)
    y = np.arange(top_probs.shape[-1])
    plt.grid()
    plt.barh(y, top_probs[i])
    plt.gca().invert_yaxis()
    plt.gca().set_axisbelow(True)
    plt.yticks(y, [cifar100.classes[index] for index in top_labels[i].numpy()])
    plt.xlabel("probability")

plt.subplots_adjust(wspace=0.5)
plt.show()

## Part III: Implicit neural field (4 points)

5. In this part, you are supposed to implement a multi-layer perceptron for implicit neural filed and train on ONE image only (overfitting).

Related lecture on implicit neural field: https://ucmercedeecs230.github.io/files/lec03.pdf

<font color="red">5.1  [1 point] </font>Implement a Multi-layer perceptron in pytorch that maps an image coordinate (x,y) to RGB color.



In [ ]:
# your code here
import torch.nn as nn
class mlp(nn.Module):
  def __init__(self):
    super().__init__()
    self.net = nn.Sequential(
    nn.Linear(2, 128),
    nn.LeakyReLU(),
    nn.BatchNorm1d(128),
    nn.Linear(128, 512),
    nn.LeakyReLU(),
    nn.BatchNorm1d(512),
    nn.Linear(512, 512),
    nn.LeakyReLU(),
    nn.BatchNorm1d(512),
    nn.Linear(512, 128),
    nn.LeakyReLU(),
    nn.BatchNorm1d(128),
    nn.Linear(128, 3),
    nn.Sigmoid()
    )
  def forward(self, x):
    return self.net(x)

<font color="red">5.2 [2 points]</font> Train the MLP for one image provided at https://ucmercedeecs230.github.io/yosemite.png , plot loss at each training iteration.

In [ ]:
# your code here

<font color="red">5.3 [1 point]</font> Once the network is trained, predict RGB color at each pixel location, visualize generated image. Also visualize difference between generated image and original image.

In [ ]:
# your code here